# Multi-View Stacking with Bayesian Meta-Learner
**Dataset:** Predict Students' Dropout and Academic Success (UCI #697)  
**Optimizations:** Optuna HPO · SMOTE-in-fold · XGBoost meta-learner · Threshold calibration · Bootstrap uncertainty

---
```
Pipeline:
  DATA (37 features)
    ├── Academic View  → [SVM, XGB★, RF★, KNN★] → 12 OOF probs
    ├── Financial View → [SVM, XGB★, RF★, KNN★] → 12 OOF probs
    ├── Demographic V. → [SVM, XGB★, RF★, KNN★] → 12 OOF probs
    └── Macro View     → [SVM, XGB★, RF★, KNN★] → 12 OOF probs
                                                     ↓
                                            48 meta-features
                                                     ↓
                                 XGBoost Meta-Learner (early stopping)
                                      ↓              ↓
                               Prediction        Bootstrap
                                                 Uncertainty
  ★ = Optuna-tuned hyperparameters
```

## 0. Install & Import

In [ ]:
# ── Install (run once) ────────────────────────────────────────────────────────
# !pip install ucimlrepo optuna xgboost imbalanced-learn shap lime scikit-learn pandas numpy matplotlib seaborn pymc --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time, json

# ── Data ─────────────────────────────────────────────────────────────────────
from ucimlrepo import fetch_ucirepo

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.svm              import SVC
from sklearn.ensemble         import RandomForestClassifier
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.metrics          import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score
)

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Imbalanced ────────────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline      import Pipeline as ImbPipeline

# ── Optuna ────────────────────────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── SHAP / LIME ───────────────────────────────────────────────────────────────
import shap
from lime.lime_tabular import LimeTabularExplainer

np.random.seed(42)
print('✅ All imports OK')

## 1. Load & Prepare Data

In [ ]:
# ── Fetch from UCI ────────────────────────────────────────────────────────────
dataset = fetch_ucirepo(id=697)
X_raw   = dataset.data.features
y_raw   = dataset.data.targets

print('Raw X shape:', X_raw.shape)
print('Raw y shape:', y_raw.shape)
print('\nTarget distribution:')
print(y_raw.value_counts(normalize=True).round(3))

In [ ]:
# ── Encode target ─────────────────────────────────────────────────────────────
# Classes: Dropout=0, Enrolled=1, Graduate=2
le = LabelEncoder()
y  = le.fit_transform(y_raw.values.ravel())
print('Label mapping:', dict(enumerate(le.classes_)))

# ── Clean column names ────────────────────────────────────────────────────────
X_raw.columns = (
    X_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)
print('\nFeatures (cleaned):')
print(list(X_raw.columns))

In [ ]:
# ── Handle missing values ─────────────────────────────────────────────────────
print('Missing values per column:')
print(X_raw.isnull().sum()[X_raw.isnull().sum() > 0])

# Fill numeric NaN with median
for col in X_raw.columns:
    if X_raw[col].isnull().any():
        X_raw[col] = X_raw[col].fillna(X_raw[col].median())

X = X_raw.values.astype(np.float32)
feature_names = list(X_raw.columns)
print(f'\nFinal X: {X.shape}, y: {y.shape}')
print('Class counts:', Counter(y))

## 2. Define Views

In [ ]:
# ── Map feature names to column indices ───────────────────────────────────────
def get_indices(names, feature_list):
    """Return indices for features whose name contains any substring in `names`."""
    idx = []
    for i, f in enumerate(feature_list):
        for n in names:
            if n.lower() in f.lower():
                idx.append(i)
                break
    return idx

# ── View definitions ──────────────────────────────────────────────────────────
VIEW_SPECS = {
    'Academic': [
        'curricular_units_1st_sem_enrolled',
        'curricular_units_1st_sem_approved',
        'curricular_units_1st_sem_grade',
        'curricular_units_1st_sem_evaluations',
        'curricular_units_1st_sem_without_evaluations',
        'curricular_units_2nd_sem_enrolled',
        'curricular_units_2nd_sem_approved',
        'curricular_units_2nd_sem_grade',
        'curricular_units_2nd_sem_evaluations',
        'curricular_units_2nd_sem_without_evaluations',
        'previous_qualification_grade',
        'admission_grade',
        'curricular_units_1st_sem_credited',
        'curricular_units_2nd_sem_credited',
    ],
    'Financial': [
        'tuition_fees_up_to_date',
        'scholarship_holder',
        'debtor',
    ],
    'Demographic': [
        'age_at_enrollment',
        'gender',
        'marital_status',
        'nacionality',  # original spelling
        'displaced',
        'international',
        'educational_special_needs',
        'mother_s_qualification',
        'father_s_qualification',
        'mother_s_occupation',
        'father_s_occupation',
        'previous_qualification',
        'application_mode',
        'application_order',
        'daytime_evening_attendance',
        'course',
    ],
    'Macro': [
        'gdp',
        'unemployment_rate',
        'inflation_rate',
    ],
}

# ── Build actual index lists from cleaned column names ────────────────────────
VIEW_INDICES = {}
for view_name, keywords in VIEW_SPECS.items():
    indices = []
    for kw in keywords:
        # Try exact match first, then partial
        for i, f in enumerate(feature_names):
            if kw in f or f in kw:
                if i not in indices:
                    indices.append(i)
    VIEW_INDICES[view_name] = sorted(indices)
    print(f'{view_name:12s}: {len(indices):2d} features → {[feature_names[i] for i in indices]}')

# ── Sanity check: any feature uncovered? ─────────────────────────────────────
all_covered = set()
for idx in VIEW_INDICES.values():
    all_covered.update(idx)

uncovered = [feature_names[i] for i in range(len(feature_names)) if i not in all_covered]
if uncovered:
    print(f'\n⚠️  Uncovered features: {uncovered}')
    # Assign uncovered to Demographic view as catch-all
    VIEW_INDICES['Demographic'] += [i for i in range(len(feature_names)) if i not in all_covered]
    VIEW_INDICES['Demographic'] = sorted(set(VIEW_INDICES['Demographic']))
    print(f'   → Added to Demographic view')
else:
    print('\n✅ All features covered across views')

## 3. Optuna Hyperparameter Tuning

In [ ]:
# ── Tune on Academic view (most complex, acts as representative) ─────────────
X_tune = X[:, VIEW_INDICES['Academic']]
y_tune = y.copy()

# Use a fixed 3-fold CV for tuning (fast)
cv_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def cv_f1_macro(estimator, X, y, cv=cv_tune):
    """Mean macro-F1 across folds (with SMOTE inside each fold)."""
    scores = []
    for tr, val in cv.split(X, y):
        X_tr, X_val = X[tr], X[val]
        y_tr, y_val = y[tr], y[val]
        sm = SMOTE(random_state=42)
        X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)
        estimator.fit(X_tr_res, y_tr_res)
        preds = estimator.predict(X_val)
        scores.append(f1_score(y_val, preds, average='macro'))
    return np.mean(scores)

print('Tuning setup ready. Starting Optuna studies...')

In [ ]:
# ── 3A. Tune XGBoost ──────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 600),
        max_depth         = trial.suggest_int('max_depth', 3, 9),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        min_child_weight  = trial.suggest_int('min_child_weight', 1, 10),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        gamma             = trial.suggest_float('gamma', 0, 5),
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = 42,
        tree_method       = 'hist',
        n_jobs            = -1,
    )
    model = XGBClassifier(**params)
    return cv_f1_macro(model, X_tune, y_tune)

t0 = time.time()
study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_objective, n_trials=50, show_progress_bar=True)
print(f'\n✅ XGB tuning done in {time.time()-t0:.0f}s')
print(f'   Best macro-F1: {study_xgb.best_value:.4f}')
print(f'   Best params: {study_xgb.best_params}')
BEST_XGB = study_xgb.best_params

In [ ]:
# ── 3B. Tune Random Forest ────────────────────────────────────────────────────
def rf_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 100, 500),
        max_depth        = trial.suggest_int('max_depth', 4, 20),
        min_samples_split= trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10),
        max_features     = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7]),
        class_weight     = 'balanced',
        random_state     = 42,
        n_jobs           = -1,
    )
    model = RandomForestClassifier(**params)
    return cv_f1_macro(model, X_tune, y_tune)

t0 = time.time()
study_rf = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(rf_objective, n_trials=30, show_progress_bar=True)
print(f'\n✅ RF tuning done in {time.time()-t0:.0f}s')
print(f'   Best macro-F1: {study_rf.best_value:.4f}')
BEST_RF = study_rf.best_params

In [ ]:
# ── 3C. Tune KNN (fast) ───────────────────────────────────────────────────────
# KNN needs scaled features
from sklearn.preprocessing import StandardScaler
scaler_tune = StandardScaler()
X_tune_scaled = scaler_tune.fit_transform(X_tune)

def knn_objective(trial):
    params = dict(
        n_neighbors  = trial.suggest_int('n_neighbors', 3, 25),
        weights      = trial.suggest_categorical('weights', ['uniform', 'distance']),
        metric       = trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski']),
        p            = trial.suggest_int('p', 1, 3),
        n_jobs       = -1,
    )
    model = KNeighborsClassifier(**params)
    return cv_f1_macro(model, X_tune_scaled, y_tune)

t0 = time.time()
study_knn = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_knn.optimize(knn_objective, n_trials=20, show_progress_bar=True)
print(f'\n✅ KNN tuning done in {time.time()-t0:.0f}s')
print(f'   Best macro-F1: {study_knn.best_value:.4f}')
BEST_KNN = study_knn.best_params

# ── SVM: fix params (tuning SVM is very slow on full dataset) ─────────────────
BEST_SVM = dict(C=2.0, gamma='scale', kernel='rbf', probability=True,
                class_weight='balanced', random_state=42)
print('\n✅ SVM params fixed (C=2.0, gamma=scale) — tuning skipped for speed')

In [ ]:
# ── Optuna importance plot for XGBoost ────────────────────────────────────────
fig = optuna.visualization.matplotlib.plot_param_importances(study_xgb)
plt.title('Optuna — XGBoost Hyperparameter Importance')
plt.tight_layout()
plt.savefig('optuna_xgb_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. OOF Training with SMOTE (5-Fold)

In [ ]:
# ── Build base model factory using tuned params ───────────────────────────────
def make_base_models():
    """Return fresh instances of 4 base models with tuned hyperparameters."""
    xgb_params = {**BEST_XGB,
                  'use_label_encoder': False,
                  'eval_metric': 'mlogloss',
                  'random_state': 42,
                  'tree_method': 'hist',
                  'n_jobs': -1}
    rf_params  = {**BEST_RF,
                  'class_weight': 'balanced',
                  'random_state': 42,
                  'n_jobs': -1}
    knn_params = {**BEST_KNN, 'n_jobs': -1}

    return {
        'SVM' : SVC(**BEST_SVM),
        'XGB' : XGBClassifier(**xgb_params),
        'RF'  : RandomForestClassifier(**rf_params),
        'KNN' : KNeighborsClassifier(**knn_params),
    }

N_CLASSES   = 3
N_MODELS    = 4
N_VIEWS     = len(VIEW_INDICES)
VIEW_NAMES  = list(VIEW_INDICES.keys())
MODEL_NAMES = ['SVM', 'XGB', 'RF', 'KNN']

# OOF meta-features: shape (n_samples, n_views × n_models × n_classes)
# = (4424, 4 × 4 × 3) = (4424, 48)
meta_train  = np.zeros((len(X), N_VIEWS * N_MODELS * N_CLASSES))
oof_preds   = np.zeros(len(X), dtype=int)

# Store trained models for later inference
fitted_models = {view: {model: [] for model in MODEL_NAMES} for view in VIEW_NAMES}
fitted_scalers = {view: [] for view in VIEW_NAMES}  # per fold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Starting OOF training: {N_VIEWS} views × {N_MODELS} models × 5 folds = {N_VIEWS*N_MODELS*5} model fits\n')

In [ ]:
# ── Main OOF Loop ─────────────────────────────────────────────────────────────
fold_scores = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n[Fold {fold_idx+1}/5] ─────────────────────────────────')

    X_tr_raw, X_val_raw = X[train_idx], X[val_idx]
    y_tr,     y_val     = y[train_idx], y[val_idx]

    view_fold_probs_val = []  # will be (n_val, 48)

    for view_name, view_idx in VIEW_INDICES.items():
        X_tr_v  = X_tr_raw[:, view_idx]
        X_val_v = X_val_raw[:, view_idx]

        # ── Scale ─────────────────────────────────────────────────────────────
        scaler_v = StandardScaler()
        X_tr_sc  = scaler_v.fit_transform(X_tr_v)
        X_val_sc = scaler_v.transform(X_val_v)

        # ── SMOTE on training fold ─────────────────────────────────────────────
        # Use min_samples strategy: k_neighbors must be < minority class count
        min_count = min(Counter(y_tr).values())
        k_smote   = min(5, min_count - 1) if min_count > 1 else 1
        sm        = SMOTE(k_neighbors=k_smote, random_state=42)
        X_tr_res, y_tr_res = sm.fit_resample(X_tr_sc, y_tr)

        base_models = make_base_models()
        view_probs  = []

        for model_name, model in base_models.items():
            # KNN already scales, but we use the same scaled data for all
            model.fit(X_tr_res, y_tr_res)
            probs = model.predict_proba(X_val_sc)   # (n_val, 3)
            view_probs.append(probs)                # 4 × (n_val, 3)

            # Store model for full-data refit later
            if fold_idx == 0:  # We'll refit on full data; this is just for reference
                pass

        # view_probs: list of 4 arrays (n_val, 3) → concat → (n_val, 12)
        view_probs_concat = np.hstack(view_probs)
        view_fold_probs_val.append(view_probs_concat)

    # Stack all views: (n_val, 48)
    meta_val = np.hstack(view_fold_probs_val)
    meta_train[val_idx] = meta_val

    # Quick fold diagnostic using simple argmax of mean probs across models
    fold_pred = np.argmax(meta_val.reshape(len(val_idx), N_VIEWS * N_MODELS, N_CLASSES).mean(axis=1), axis=1)
    fold_f1   = f1_score(y_val, fold_pred, average='macro')
    fold_scores.append(fold_f1)
    print(f'  OOF macro-F1 (argmax ensemble): {fold_f1:.4f}')

print(f'\n✅ OOF Complete!')
print(f'   Mean macro-F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'   meta_train shape: {meta_train.shape}')

## 5. Refit Base Models on Full Training Data

In [ ]:
# ── Refit all base models on FULL data (for test-time inference) ─────────────
# Also build meta_test from a held-out test split
from sklearn.model_selection import train_test_split

# 80/20 split for final evaluation
X_full_tr, X_test, y_full_tr, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Full train: {X_full_tr.shape}, Test: {X_test.shape}')

# ── Build meta_test features using base models trained on full train ───────────
full_base_models = {}   # view → model_name → fitted_model
full_scalers     = {}   # view → scaler

meta_test = np.zeros((len(X_test), N_VIEWS * N_MODELS * N_CLASSES))

# Also rebuild meta_train for meta-learner training using 80% split
meta_train_final = np.zeros((len(X_full_tr), N_VIEWS * N_MODELS * N_CLASSES))

skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print('\nBuilding meta_train_final via inner CV on 80% split...')
for fold_idx, (tr_idx, val_idx) in enumerate(skf_inner.split(X_full_tr, y_full_tr)):
    X_tr_raw, X_val_raw = X_full_tr[tr_idx], X_full_tr[val_idx]
    y_tr_,    y_val_    = y_full_tr[tr_idx],  y_full_tr[val_idx]

    view_probs_val = []
    for view_name, view_idx in VIEW_INDICES.items():
        X_tr_v  = X_tr_raw[:, view_idx]
        X_val_v = X_val_raw[:, view_idx]
        scaler_v = StandardScaler()
        X_tr_sc  = scaler_v.fit_transform(X_tr_v)
        X_val_sc = scaler_v.transform(X_val_v)

        min_count = min(Counter(y_tr_).values())
        k_smote   = min(5, min_count - 1) if min_count > 1 else 1
        sm        = SMOTE(k_neighbors=k_smote, random_state=42)
        X_tr_res, y_tr_res = sm.fit_resample(X_tr_sc, y_tr_)

        models_v  = make_base_models()
        view_probs = []
        for model_name, model in models_v.items():
            model.fit(X_tr_res, y_tr_res)
            view_probs.append(model.predict_proba(X_val_sc))
        view_probs_val.append(np.hstack(view_probs))

    meta_train_final[val_idx] = np.hstack(view_probs_val)

print('Building meta_test from full 80% trained models...')
for view_name, view_idx in VIEW_INDICES.items():
    X_tr_v   = X_full_tr[:, view_idx]
    X_test_v = X_test[:, view_idx]
    scaler_v = StandardScaler()
    X_tr_sc  = scaler_v.fit_transform(X_tr_v)
    X_test_sc = scaler_v.transform(X_test_v)
    full_scalers[view_name] = scaler_v

    min_count = min(Counter(y_full_tr).values())
    k_smote   = min(5, min_count - 1) if min_count > 1 else 1
    sm        = SMOTE(k_neighbors=k_smote, random_state=42)
    X_tr_res, y_tr_res = sm.fit_resample(X_tr_sc, y_full_tr)

    models_v    = make_base_models()
    full_base_models[view_name] = {}
    col_offset  = VIEW_NAMES.index(view_name) * N_MODELS * N_CLASSES
    view_probs  = []
    for model_name, model in models_v.items():
        model.fit(X_tr_res, y_tr_res)
        full_base_models[view_name][model_name] = model
        view_probs.append(model.predict_proba(X_test_sc))

    meta_test[:, col_offset:col_offset + N_MODELS*N_CLASSES] = np.hstack(view_probs)

print(f'\n✅ meta_train_final: {meta_train_final.shape}')
print(f'   meta_test:        {meta_test.shape}')

## 6. XGBoost Meta-Learner with Early Stopping

In [ ]:
# ── Meta-learner column names ──────────────────────────────────────────────────
meta_col_names = [
    f'{view}_{model}_c{cls}'
    for view  in VIEW_NAMES
    for model in MODEL_NAMES
    for cls   in range(N_CLASSES)
]
print(f'Meta-feature columns ({len(meta_col_names)}): {meta_col_names[:6]}...')

In [ ]:
# ── Optuna: tune meta-learner XGBoost ─────────────────────────────────────────
from sklearn.model_selection import cross_val_score

def meta_xgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 800),
        max_depth         = trial.suggest_int('max_depth', 2, 7),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 0.5, 5.0),
        min_child_weight  = trial.suggest_int('min_child_weight', 1, 8),
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = 42,
        tree_method       = 'hist',
        n_jobs            = -1,
    )
    model = XGBClassifier(**params)
    scores = cross_val_score(model, meta_train_final, y_full_tr,
                              cv=3, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

t0 = time.time()
study_meta = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_meta.optimize(meta_xgb_objective, n_trials=40, show_progress_bar=True)
print(f'\n✅ Meta XGB tuning done in {time.time()-t0:.0f}s')
print(f'   Best macro-F1 (CV): {study_meta.best_value:.4f}')
BEST_META = study_meta.best_params

In [ ]:
# ── Train final meta-learner with early stopping ──────────────────────────────
from sklearn.model_selection import train_test_split as tts

# Keep 15% of meta_train as internal validation for early stopping
M_tr, M_val, yM_tr, yM_val = tts(
    meta_train_final, y_full_tr,
    test_size=0.15, stratify=y_full_tr, random_state=42
)

meta_learner = XGBClassifier(
    **{**BEST_META,
       'use_label_encoder': False,
       'eval_metric': 'mlogloss',
       'random_state': 42,
       'tree_method': 'hist',
       'n_jobs': -1,
       'early_stopping_rounds': 30,
    }
)

meta_learner.fit(
    M_tr, yM_tr,
    eval_set=[(M_val, yM_val)],
    verbose=50
)

print(f'\nBest iteration: {meta_learner.best_iteration}')

## 7. Threshold Optimization

In [ ]:
# ── Optimize decision thresholds to maximise macro-F1 on validation set ───────
# Use the OOF meta predictions as a proxy validation set
oof_meta_probs = meta_learner.predict_proba(meta_train_final)  # (n_train, 3)

from scipy.optimize import minimize

def threshold_predict(probs, thresholds):
    """Apply per-class thresholds: assign class with max(prob - threshold)."""
    adjusted = probs - np.array(thresholds)
    return np.argmax(adjusted, axis=1)

def neg_f1_macro(thresholds, probs, labels):
    preds = threshold_predict(probs, thresholds)
    return -f1_score(labels, preds, average='macro')

# Initial: uniform thresholds
init_thresholds = [1/3, 1/3, 1/3]
result = minimize(
    neg_f1_macro,
    init_thresholds,
    args=(oof_meta_probs, y_full_tr),
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-5, 'fatol': 1e-5}
)

OPTIMAL_THRESHOLDS = result.x
print(f'Optimal thresholds: Dropout={OPTIMAL_THRESHOLDS[0]:.4f}, '
      f'Enrolled={OPTIMAL_THRESHOLDS[1]:.4f}, Graduate={OPTIMAL_THRESHOLDS[2]:.4f}')

# Compare default vs threshold-optimized on training set
pred_default = np.argmax(oof_meta_probs, axis=1)
pred_thresh  = threshold_predict(oof_meta_probs, OPTIMAL_THRESHOLDS)

f1_default = f1_score(y_full_tr, pred_default, average='macro')
f1_thresh  = f1_score(y_full_tr, pred_thresh,  average='macro')
print(f'\nOOF macro-F1 — default: {f1_default:.4f} | threshold-optimized: {f1_thresh:.4f}')

## 8. Evaluation on Test Set

In [ ]:
# ── Test set predictions ───────────────────────────────────────────────────────
test_meta_probs = meta_learner.predict_proba(meta_test)  # (n_test, 3)
test_preds      = threshold_predict(test_meta_probs, OPTIMAL_THRESHOLDS)

class_names = le.classes_
print('=' * 60)
print('CLASSIFICATION REPORT — Test Set')
print('=' * 60)
print(classification_report(y_test, test_preds, target_names=class_names))

acc = accuracy_score(y_test, test_preds)
f1  = f1_score(y_test, test_preds, average='macro')
auc = roc_auc_score(y_test, test_meta_probs, multi_class='ovr', average='macro')

print(f'Accuracy:       {acc:.4f}')
print(f'Macro-F1:       {f1:.4f}')
print(f'ROC-AUC (OvR):  {auc:.4f}')

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Confusion Matrix (Counts)', 'Confusion Matrix (Normalized)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

plt.suptitle('Multi-View Stacking — Test Set Results', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Bootstrap Uncertainty Estimation

In [ ]:
# ── Bootstrap over base models to estimate prediction uncertainty ─────────────
# Instead of retraining (slow), we sample rows of meta_test to simulate noise
# and report variance across bootstrap rounds.
#
# For a more principled approach: retrain meta-learner on B bootstrap samples
# of meta_train_final and collect predictions.

N_BOOTSTRAP = 100
bootstrap_preds = np.zeros((N_BOOTSTRAP, len(X_test), N_CLASSES))

rng = np.random.RandomState(42)
n_tr = len(meta_train_final)

print(f'Running {N_BOOTSTRAP} bootstrap iterations...')
t0 = time.time()

for b in range(N_BOOTSTRAP):
    # Sample training indices with replacement
    boot_idx = rng.choice(n_tr, size=n_tr, replace=True)
    X_boot   = meta_train_final[boot_idx]
    y_boot   = y_full_tr[boot_idx]

    # Lightweight meta-learner for bootstrap (fewer trees)
    boot_params = {**BEST_META,
                   'use_label_encoder': False,
                   'eval_metric': 'mlogloss',
                   'random_state': b,
                   'tree_method': 'hist',
                   'n_jobs': -1}
    # Remove early_stopping_rounds for bootstrap (no eval set)
    boot_params.pop('early_stopping_rounds', None)

    bm = XGBClassifier(**boot_params)
    bm.fit(X_boot, y_boot)
    bootstrap_preds[b] = bm.predict_proba(meta_test)

print(f'Done in {time.time()-t0:.0f}s')

# ── Compute uncertainty metrics ───────────────────────────────────────────────
mean_probs   = bootstrap_preds.mean(axis=0)          # (n_test, 3)
std_probs    = bootstrap_preds.std(axis=0)           # (n_test, 3)
ci_low       = np.percentile(bootstrap_preds, 2.5, axis=0)   # (n_test, 3)
ci_high      = np.percentile(bootstrap_preds, 97.5, axis=0)  # (n_test, 3)

# Uncertainty score = std of the predicted class probability
pred_class     = np.argmax(mean_probs, axis=1)
uncertainty    = std_probs[np.arange(len(X_test)), pred_class]

print(f'\nMean uncertainty (std): {uncertainty.mean():.4f}')
print(f'Max uncertainty:        {uncertainty.max():.4f}')
print(f'Samples with uncertainty > 0.10: {(uncertainty > 0.10).sum()}')

In [ ]:
# ── Uncertainty analysis plots ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Distribution of uncertainty
axes[0].hist(uncertainty, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(0.10, color='red', linestyle='--', label='High uncertainty threshold')
axes[0].set_xlabel('Prediction Uncertainty (std)')
axes[0].set_ylabel('Count')
axes[0].set_title('Uncertainty Distribution')
axes[0].legend()

# Plot 2: Accuracy vs uncertainty bucket
bins    = [0, 0.05, 0.10, 0.15, 0.20, 1.0]
labels_ = ['<0.05', '0.05-0.10', '0.10-0.15', '0.15-0.20', '>0.20']
bucket  = pd.cut(uncertainty, bins=bins, labels=labels_)

correct = (pred_class == y_test).astype(int)
bucket_acc = pd.DataFrame({'bucket': bucket, 'correct': correct}).groupby('bucket')['correct'].mean()
bucket_acc.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_xlabel('Uncertainty Bucket')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy by Uncertainty Level')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=45)

# Plot 3: CI width distribution per class
ci_width = ci_high - ci_low  # (n_test, 3)
df_ci = pd.DataFrame(ci_width, columns=class_names)
df_ci.boxplot(ax=axes[2])
axes[2].set_xlabel('Class')
axes[2].set_ylabel('95% CI Width')
axes[2].set_title('Bootstrap CI Width per Class')

plt.suptitle('Uncertainty Analysis — Bootstrap (n=100)', fontsize=14)
plt.tight_layout()
plt.savefig('uncertainty_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── High-confidence vs low-confidence accuracy ────────────────────────────────
HIGH_CONF_THRESH = 0.08
high_conf_mask   = uncertainty <= HIGH_CONF_THRESH
low_conf_mask    = uncertainty >  HIGH_CONF_THRESH

print(f'High-confidence samples (σ ≤ {HIGH_CONF_THRESH}): {high_conf_mask.sum()}')
print(f'  Accuracy: {accuracy_score(y_test[high_conf_mask], pred_class[high_conf_mask]):.4f}')
print(f'\nLow-confidence samples (σ > {HIGH_CONF_THRESH}): {low_conf_mask.sum()}')
if low_conf_mask.sum() > 0:
    print(f'  Accuracy: {accuracy_score(y_test[low_conf_mask], pred_class[low_conf_mask]):.4f}')

## 10. SHAP Analysis — Level 1 (Base Models)

In [ ]:
# ── SHAP Level 1: XGBoost per view on full training data ─────────────────────
print('Computing SHAP values for XGBoost base models...')

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

shap_per_view = {}  # store for later

for idx, (view_name, view_idx) in enumerate(VIEW_INDICES.items()):
    model_xgb = full_base_models[view_name]['XGB']
    scaler_v  = full_scalers[view_name]

    X_test_v   = X_test[:, view_idx]
    X_test_sc  = scaler_v.transform(X_test_v)
    feat_names = [feature_names[i] for i in view_idx]

    explainer   = shap.TreeExplainer(model_xgb)
    shap_values = explainer.shap_values(X_test_sc)
    shap_per_view[view_name] = (shap_values, X_test_sc, feat_names)

    # Mean absolute SHAP across classes
    if isinstance(shap_values, list):
        mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    else:
        mean_shap = np.abs(shap_values).mean(axis=0)

    # Sort and plot
    sorted_idx  = np.argsort(mean_shap)[::-1][:10]  # top 10
    top_names   = [feat_names[i] for i in sorted_idx]
    top_values  = mean_shap[sorted_idx]

    axes[idx].barh(top_names[::-1], top_values[::-1], color='#4C72B0')
    axes[idx].set_xlabel('Mean |SHAP Value|')
    axes[idx].set_title(f'SHAP — {view_name} View (XGBoost)', fontsize=12)
    axes[idx].tick_params(labelsize=9)

plt.suptitle('SHAP Level 1: Feature Importance per View', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('shap_level1_per_view.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP beeswarm for Academic View (most important) ─────────────────────────
shap_values_acad, X_acad_sc, feat_acad = shap_per_view['Academic']

# For multi-class: show Dropout class (index 0)
if isinstance(shap_values_acad, list):
    sv_dropout = shap_values_acad[0]
else:
    sv_dropout = shap_values_acad

plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_dropout, X_acad_sc,
    feature_names=feat_acad,
    show=False, max_display=12
)
plt.title('SHAP Beeswarm — Academic View (Dropout class)', fontsize=13)
plt.tight_layout()
plt.savefig('shap_beeswarm_academic.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. SHAP Analysis — Level 2 (Meta-Learner)

In [ ]:
# ── SHAP Level 2: which view/model combination drives the meta-prediction ─────
explainer_meta  = shap.TreeExplainer(meta_learner)
shap_meta       = explainer_meta.shap_values(meta_test)  # list of 3 arrays or array

# Aggregate by view
view_shap_scores = {}

for v_idx, view_name in enumerate(VIEW_NAMES):
    col_start = v_idx * N_MODELS * N_CLASSES
    col_end   = col_start + N_MODELS * N_CLASSES

    if isinstance(shap_meta, list):
        # Multi-class: average over classes
        view_abs = np.mean([
            np.abs(sm[:, col_start:col_end]).mean()
            for sm in shap_meta
        ])
    else:
        view_abs = np.abs(shap_meta[:, col_start:col_end]).mean()

    view_shap_scores[view_name] = view_abs

# Normalize
total = sum(view_shap_scores.values())
for k in view_shap_scores:
    view_shap_scores[k] /= total

print('View importance (normalized mean |SHAP|):')
for k, v in sorted(view_shap_scores.items(), key=lambda x: -x[1]):
    print(f'  {k:12s}: {v:.4f} ({v*100:.1f}%)')

In [ ]:
# ── Plot view contributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# View-level bar chart
sorted_views = sorted(view_shap_scores.items(), key=lambda x: -x[1])
axes[0].barh([v[0] for v in sorted_views][::-1],
             [v[1] for v in sorted_views][::-1],
             color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'])
axes[0].set_xlabel('Normalized Mean |SHAP|')
axes[0].set_title('SHAP Level 2: View Contribution to Meta-Learner')

# Model-level breakdown across all views
model_shap_scores = {}
for m_idx, model_name in enumerate(MODEL_NAMES):
    model_cols = []
    for v_idx in range(N_VIEWS):
        base = v_idx * N_MODELS * N_CLASSES + m_idx * N_CLASSES
        model_cols.extend(range(base, base + N_CLASSES))

    if isinstance(shap_meta, list):
        mv = np.mean([np.abs(sm[:, model_cols]).mean() for sm in shap_meta])
    else:
        mv = np.abs(shap_meta[:, model_cols]).mean()
    model_shap_scores[model_name] = mv

total_m = sum(model_shap_scores.values())
for k in model_shap_scores:
    model_shap_scores[k] /= total_m

sorted_models = sorted(model_shap_scores.items(), key=lambda x: -x[1])
axes[1].barh([v[0] for v in sorted_models][::-1],
             [v[1] for v in sorted_models][::-1],
             color=['#F44336', '#3F51B5', '#009688', '#FF5722'])
axes[1].set_xlabel('Normalized Mean |SHAP|')
axes[1].set_title('SHAP Level 2: Model Type Contribution')

plt.suptitle('SHAP Level 2: Meta-Learner Explainability', fontsize=14)
plt.tight_layout()
plt.savefig('shap_level2_meta.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. LIME — Individual Student Explanation

In [ ]:
# ── LIME on a single student (meta-feature space) ─────────────────────────────
lime_explainer = LimeTabularExplainer(
    meta_train_final,
    feature_names=meta_col_names,
    class_names=list(class_names),
    mode='classification',
    random_state=42
)

# Pick a high-uncertainty student as example
student_idx = np.argmax(uncertainty)
print(f'Explaining student at test index: {student_idx}')
print(f'True class:      {class_names[y_test[student_idx]]}')
print(f'Predicted class: {class_names[pred_class[student_idx]]}')
print(f'Uncertainty σ:   {uncertainty[student_idx]:.4f}')
print(f'Class probs:     {mean_probs[student_idx].round(4)}')
print(f'95% CI:          [{ci_low[student_idx].round(4)}, {ci_high[student_idx].round(4)}]')

In [ ]:
exp = lime_explainer.explain_instance(
    meta_test[student_idx],
    meta_learner.predict_proba,
    num_features=10,
    labels=[pred_class[student_idx]]
)

fig = exp.as_pyplot_figure(label=pred_class[student_idx])
plt.title(f'LIME — Student #{student_idx} (Predicted: {class_names[pred_class[student_idx]]}, '
          f'σ={uncertainty[student_idx]:.3f})', fontsize=11)
plt.tight_layout()
plt.savefig('lime_student.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Final Results Summary

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score

results = {
    'Model'        : 'Multi-View Stacking (XGB Meta)',
    'Accuracy'     : round(accuracy_score(y_test, test_preds), 4),
    'Macro F1'     : round(f1_score(y_test, test_preds, average='macro'), 4),
    'Macro Prec.'  : round(precision_score(y_test, test_preds, average='macro'), 4),
    'Macro Recall' : round(recall_score(y_test, test_preds, average='macro'), 4),
    'ROC-AUC'      : round(roc_auc_score(y_test, test_meta_probs, multi_class='ovr', average='macro'), 4),
    'OOF CV F1'    : round(np.mean(fold_scores), 4),
    'High-Conf Acc': round(accuracy_score(y_test[high_conf_mask], pred_class[high_conf_mask]), 4),
    'Thresholds'   : [round(t, 4) for t in OPTIMAL_THRESHOLDS],
}

print('\n' + '='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
for k, v in results.items():
    print(f'  {k:<20}: {v}')
print('='*60)

# View importance ranking
print('\nView Importance (SHAP Level 2):')
for view, score in sorted(view_shap_scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 40)
    print(f'  {view:12s} {bar} {score*100:.1f}%')

In [ ]:
# ── Save results to JSON (for paper/thesis reporting) ─────────────────────────
output = {
    'results'           : results,
    'view_importance'   : {k: round(v, 4) for k, v in view_shap_scores.items()},
    'model_importance'  : {k: round(v, 4) for k, v in model_shap_scores.items()},
    'optimal_thresholds': [round(t, 4) for t in OPTIMAL_THRESHOLDS],
    'best_xgb_base'     : {k: (float(v) if hasattr(v,'item') else v) for k, v in BEST_XGB.items()},
    'best_rf_base'      : {k: (float(v) if hasattr(v,'item') else v) for k, v in BEST_RF.items()},
    'best_knn_base'     : {k: (float(v) if hasattr(v,'item') else v) for k, v in BEST_KNN.items()},
    'best_meta_xgb'     : {k: (float(v) if hasattr(v,'item') else v) for k, v in BEST_META.items()},
}

with open('multiview_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('✅ Results saved to multiview_results.json')
print('\nFiles produced:')
import os
for fn in sorted(os.listdir('.')):
    if fn.endswith(('.png', '.json')):
        print(f'  {fn}')